# GPTQ quantization

GPTQ is a post-training, weight-only quantizer. The original paper introduces "accurate post-training quantization" for generative pretrained transformers using approximate second-order information.

Sources:
- Paper: https://arxiv.org/abs/2210.17323
- Original code: https://github.com/IST-DASLab/gptq
- Current HF integration: https://huggingface.co/docs/transformers/quantization/gptq

The notebook uses `transformers.GPTQConfig` with the maintained `gptqmodel` backend when available. For CPU inference later, load with `use_exllama=False` because ExLlama kernels are GPU-only.

In [ ]:
!pip -q install -U transformers accelerate optimum datasets sentencepiece safetensors
!pip -q install -U gptqmodel --no-build-isolation

In [ ]:

import json
import os
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def bytes_to_gib(n):
    return n / (1024 ** 3)


def directory_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def cuda_peak_gib():
    if not torch.cuda.is_available():
        return 0.0
    return bytes_to_gib(torch.cuda.max_memory_allocated())


def infer_model_device(model):
    for candidate in (model, getattr(model, "model", None)):
        if candidate is None:
            continue
        try:
            return next(candidate.parameters()).device
        except (AttributeError, StopIteration):
            pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def generate_benchmark(model, tokenizer, prompt, max_new_tokens=64, runs=3):
    device = infer_model_device(model)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    times = []
    token_count = 0
    with torch.no_grad():
        for _ in range(runs):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            times.append(elapsed)
            token_count = out.shape[-1] - inputs["input_ids"].shape[-1]
    avg = sum(times) / len(times)
    return {
        "avg_seconds": avg,
        "generated_tokens": int(token_count),
        "tokens_per_second": float(token_count / avg) if avg else 0.0,
        "cuda_peak_gib": cuda_peak_gib(),
    }


def save_metrics(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))

from transformers import GPTQConfig


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = Path("/content/quant_outputs/gptq_int4")
BITS = 4
GROUP_SIZE = 128
PROMPT = "Quantization reduces inference memory by"
MAX_NEW_TOKENS = 64
RUNS = 3

CALIBRATION_TEXTS = [
    "Large language models are compressed to reduce memory bandwidth during inference.",
    "Post training quantization uses a small calibration set and does not update weights.",
    "The benchmark should compare perplexity, latency, throughput, and peak memory.",
    "Weight only quantization stores linear layer weights in fewer bits.",
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
gptq_config = GPTQConfig(
    bits=BITS,
    group_size=GROUP_SIZE,
    dataset=CALIBRATION_TEXTS,
    tokenizer=tokenizer,
    desc_act=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    quantization_config=gptq_config,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR / "model")
tokenizer.save_pretrained(OUTPUT_DIR / "model")

metrics = generate_benchmark(model, tokenizer, PROMPT, MAX_NEW_TOKENS, RUNS)
metrics.update({
    "method": "gptq",
    "bits": BITS,
    "group_size": GROUP_SIZE,
    "model_id": MODEL_ID,
    "artifact_dir": str(OUTPUT_DIR / "model"),
    "artifact_size_gib": bytes_to_gib(directory_size_bytes(OUTPUT_DIR / "model")),
})
save_metrics(OUTPUT_DIR / "metrics.json", metrics)

In [ ]:
# Optional CPU-oriented reload. This is useful when you later test CPU loading.
cpu_config = GPTQConfig(bits=BITS, use_exllama=False)
cpu_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR / "model",
    device_map="cpu",
    quantization_config=cpu_config,
)
print(type(cpu_model))